## Primary Pricing Regression

This notebook evaluates premium rate adequacy using OLS regression with geographic, hazard, seasonality, and coverage-band features.

Key steps:
- Load and prepare pricing data
- Create feature bands and dummy variables
- Compare multiple model specifications
- Identify and review outliers
- Visualize actual vs. predicted premium rate

In [ ]:
# Full Dataset Regression Model
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Load Data
file_path = "data/sample_pricing_data.xlsx"
df = pd.read_excel(file_path, sheet_name="PRIMARY DATA")

# Prepare Data
df["DATE BOUND"] = pd.to_datetime(df["DATE BOUND"])
df["MONTH_BOUND"] = df["DATE BOUND"].dt.to_period("M").astype(str)
df["HAZARD_SCORE"] = df["FIRE SCORE"].astype(str)
df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
df["PROPERTY LIMIT"] = pd.to_numeric(df["PROPERTY LIMIT"], errors="coerce")
df["POLICY LIMIT - COV A"] = pd.to_numeric(df["POLICY LIMIT - COV A"], errors="coerce")
df["PREMIUM RATE (%)"] = pd.to_numeric(df["PREMIUM RATE (%)"], errors="coerce")

# Limit Bands
limit_bins = [0, 1e6, 5e6, 10e6, np.inf]
limit_labels = ["0-1M", "1-5M", "5-10M", ">10M"]
df["TIV_BAND"] = pd.cut(df["PROPERTY LIMIT"], bins=limit_bins, labels=limit_labels)
df["COV_A_BAND"] = pd.cut(df["POLICY LIMIT - COV A"], bins=limit_bins, labels=limit_labels)

# Optional generic segment flag
# Replace with a real project-safe feature if available in your sample dataset
if "SPECIAL_SEGMENT_FLAG" not in df.columns:
    df["SPECIAL_SEGMENT_FLAG"] = 0

df = df.dropna(subset=["PREMIUM RATE (%)", "LATITUDE", "LONGITUDE"]).copy()

print(df.head())
print(df.shape)

In [ ]:
# Generic segment list replacement
# In the original analysis, a specific policy group was excluded for comparison.
# For GitHub-safe version, use a generic flag instead of real policy numbers.

special_segment_flag = "SPECIAL_SEGMENT_FLAG"

In [ ]:
# Create 4 datasets
def create_dataset(exclude_month=False, exclude_special_segment=False):
    df_filtered = df.copy()

    if exclude_special_segment:
        df_filtered = df_filtered[df_filtered[special_segment_flag] != 1]

    y = df_filtered["PREMIUM RATE (%)"]
    X_base = df_filtered[["LATITUDE", "LONGITUDE"]]

    dummies_cols = ["HAZARD_SCORE", "TIV_BAND", "COV_A_BAND"]
    if not exclude_month:
        dummies_cols.append("MONTH_BOUND")

    X_cat = pd.get_dummies(df_filtered[dummies_cols], drop_first=False)
    X = pd.concat([X_base, X_cat], axis=1)
    X = sm.add_constant(X)

    return X.astype(float), y, df_filtered

In [ ]:
# Run all models
models = {}

labels = [
    "Full Data",
    "No Monthly Variable",
    "No Special Segment",
    "No Special Segment + No Monthly Variable"
]

settings = [
    (False, False),
    (True, False),
    (False, True),
    (True, True)
]

for label, (drop_month, drop_segment) in zip(labels, settings):
    X, y, df_use = create_dataset(drop_month, drop_segment)
    model = sm.OLS(y, X).fit()
    models[label] = model

    print(f"\n{label}")
    print("R-squared:", round(model.rsquared, 3))
    print(model.summary().tables[0])

In [ ]:
# Identify Outliers - Full Model
model_full = models["Full Data"]
X_full, y_full, df_full = create_dataset(False, False)

y_pred = model_full.predict(X_full)
resid_std = np.std(y_full - y_pred)

lower_bound = y_pred - 1.96 * resid_std
upper_bound = y_pred + 1.96 * resid_std
outliers_mask = (y_full < lower_bound) | (y_full > upper_bound)

outliers_df = df_full.loc[outliers_mask].copy()
outliers_df["Predicted PREMIUM RATE"] = y_pred[outliers_mask]
outliers_df["Residual"] = y_full[outliers_mask] - y_pred[outliers_mask]

print(outliers_df.head())
print("Outlier count:", len(outliers_df))

In [ ]:
# Plot with 95% Band + Highlight Outliers
plt.figure(figsize=(8, 6))
plt.scatter(y_full, y_pred, alpha=0.6, label="All Data")
plt.scatter(y_full[outliers_mask], y_pred[outliers_mask], color="red", label="Outliers")
plt.plot([y_full.min(), y_full.max()], [y_full.min(), y_full.max()], "r--", label="Ideal Fit")
plt.fill_between(y_full, lower_bound, upper_bound, color="gray", alpha=0.2, label="95% Band")

plt.xlabel("Actual PREMIUM RATE")
plt.ylabel("Predicted PREMIUM RATE")
plt.title("Actual vs Predicted with 95% Confidence Band")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Rebuild model excluding special segment
model_no_segment = models["No Special Segment"]
X_no_segment, y_no_segment, df_no_segment = create_dataset(False, True)

y_pred_no_segment = model_no_segment.predict(X_no_segment)
resid_std_no_segment = np.std(y_no_segment - y_pred_no_segment)

lower_bound_no_segment = y_pred_no_segment - 1.96 * resid_std_no_segment
upper_bound_no_segment = y_pred_no_segment + 1.96 * resid_std_no_segment
outliers_mask_no_segment = (y_no_segment < lower_bound_no_segment) | (y_no_segment > upper_bound_no_segment)

outliers_df_no_segment = df_no_segment.loc[outliers_mask_no_segment].copy()
outliers_df_no_segment["Predicted PREMIUM RATE"] = y_pred_no_segment[outliers_mask_no_segment]
outliers_df_no_segment["Residual"] = y_no_segment[outliers_mask_no_segment] - y_pred_no_segment[outliers_mask_no_segment]

print(outliers_df_no_segment.head())
print("Outlier count (no special segment):", len(outliers_df_no_segment))

In [ ]:
# Plot excluding special segment
plt.figure(figsize=(8, 6))
plt.scatter(y_no_segment, y_pred_no_segment, alpha=0.6, label="All Data")
plt.scatter(
    y_no_segment[outliers_mask_no_segment],
    y_pred_no_segment[outliers_mask_no_segment],
    color="red",
    label="Outliers"
)
plt.plot(
    [y_no_segment.min(), y_no_segment.max()],
    [y_no_segment.min(), y_no_segment.max()],
    "r--",
    label="Ideal Fit"
)
plt.fill_between(
    y_no_segment,
    lower_bound_no_segment,
    upper_bound_no_segment,
    color="gray",
    alpha=0.2,
    label="95% Band"
)

plt.xlabel("Actual PREMIUM RATE")
plt.ylabel("Predicted PREMIUM RATE")
plt.title("Actual vs Predicted (Excluding Special Segment)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Key Findings

- Geographic variables, hazard score, and coverage bands explain a meaningful portion of premium-rate variation.
- Monthly effects can be tested separately to evaluate seasonality impact.
- Excluding a specific segment provides a useful sensitivity check for pricing adequacy.
- Outlier review helps identify accounts that may require underwriting or pricing adjustment.